In [16]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.optimize import minimize

In [17]:
tickers = ["0002.HK", "0003.HK", "0778.HK", "0823.HK", "0941.HK", "0006.HK"] # Just example stock tickers, can be changed according to your need
data = yf.download(tickers, start="2023-01-01", end="2026-01-01")['Close']
returns = data.pct_change().dropna().to_numpy()

[*********************100%***********************]  6 of 6 completed


In [18]:
mean_returns = np.mean(returns, axis=0) * 252
cov_matrix = np.cov(returns, rowvar=False) * 252
risk_free_rate = 0.03275 # 10 year bond yield taken from https://tradingeconomics.com/hong-kong/government-bond-yield 
num_assets = len(tickers)

In [19]:
def negative_sharpe(weights):
    p_return = np.dot(weights, mean_returns)
    p_variance = np.dot(weights.T, np.dot(cov_matrix, weights))
    p_risk = np.sqrt(p_variance)
    sharpe = (p_return - risk_free_rate) / p_risk
    return -sharpe

In [20]:
constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
bounds = tuple((0, 1) for i in range(num_assets))
initial_guess = num_assets * [1.0 / num_assets]

In [21]:
optimized_result = minimize(
    fun=negative_sharpe, 
    x0=initial_guess, 
    method='SLSQP', 
    bounds=bounds, 
    constraints=constraints
)

In [22]:
best_weights = optimized_result.x
max_sharpe = -optimized_result.fun  

print("--- optimal portfolio found via scipy  ---")
for ticker, weight in zip(tickers, best_weights):
    print(f"{ticker} Exact Weight: {weight:.2%}")
print(f"Maximized Sharpe Ratio: {max_sharpe:.4f}")

--- optimal portfolio found via scipy  ---
0002.HK Exact Weight: 0.63%
0003.HK Exact Weight: 0.00%
0778.HK Exact Weight: 29.90%
0823.HK Exact Weight: 0.00%
0941.HK Exact Weight: 0.00%
0006.HK Exact Weight: 69.46%
Maximized Sharpe Ratio: 1.2549
